In [ ]:
import pandas as pd
import numpy as np
import json
import csv
import ast 
import datetime as dt
from collections import Counter, defaultdict


pd.set_option("display.max_colwidth", None)
pd.set_option("display.max_columns", None)
pd.set_option("display.width", 1000)

#first, learn the structure and the column values

CSV_PATH = "simple_process.csv"
df = pd.read_csv(CSV_PATH, header=None)
df.head()

,0,1,2,3,4,5
0,PLC/Island 1/VacuumGripper/VacuumGripper01/measurements/input/vacuumSensVerticalEncoderCounter,"{""value"": 1770, ""timestamp"": ""2025-05-12T11:59:13.992112+00:00Z""}",0,True,1.747051e+09,0.040951
1,PLC/Island 1/VacuumGripper/VacuumGripper01/measurements/input/vacuumSensRotEncoderCounter,"{""value"": 2690, ""timestamp"": ""2025-05-12T11:59:13.992289+00:00Z""}",0,True,1.747051e+09,0.000252
2,PLC/Island 1/VacuumGripper/VacuumGripper01/measurements/input/vacuumSensArmEncoderCounter,"{""value"": 909, ""timestamp"": ""2025-05-12T11:59:13.992372+00:00Z""}",0,True,1.747051e+09,0.000188
3,PLC/Island 1/VacuumGripper/VacuumGripper01/measurements/input/vacuumSensVerticalEndUp,"{""value"": false, ""timestamp"": ""2025-05-12T11:59:13.992448+00:00Z""}",0,True,1.747051e+09,0.000135
4,PLC/Island 1/VacuumGripper/VacuumGripper01/measurements/input/vacuumSensRotEnd,"{""value"": false, ""timestamp"": ""2025-05-12T11:59:13.992523+00:00Z""}",0,True,1.747051e+09,0.000130


In [3]:
df.describe

<bound method NDFrame.describe of                                                                                                     0                                                                   1  2      3             4         5
0      PLC/Island 1/VacuumGripper/VacuumGripper01/measurements/input/vacuumSensVerticalEncoderCounter   {"value": 1770, "timestamp": "2025-05-12T11:59:13.992112+00:00Z"}  0   True  1.747051e+09  0.040951
1           PLC/Island 1/VacuumGripper/VacuumGripper01/measurements/input/vacuumSensRotEncoderCounter   {"value": 2690, "timestamp": "2025-05-12T11:59:13.992289+00:00Z"}  0   True  1.747051e+09  0.000252
2           PLC/Island 1/VacuumGripper/VacuumGripper01/measurements/input/vacuumSensArmEncoderCounter    {"value": 909, "timestamp": "2025-05-12T11:59:13.992372+00:00Z"}  0   True  1.747051e+09  0.000188
3               PLC/Island 1/VacuumGripper/VacuumGripper01/measurements/input/vacuumSensVerticalEndUp  {"value": false, "timestamp": "2025-05-12T11:59

In [4]:
for c in df.columns:
    print(f"col {c}: dtype={df[c].dtype}, n_unique={df[c].nunique()}")

col 0: dtype=object, n_unique=118
col 1: dtype=object, n_unique=31911
col 2: dtype=int64, n_unique=1
col 3: dtype=bool, n_unique=2
col 4: dtype=float64, n_unique=31911
col 5: dtype=float64, n_unique=10888


In [5]:
print("unique values in column 2: ", df[2].unique)
print("value count in column 3: ", df[2].value_counts())
print("unique values in column 3: ", df[3].unique)
print("value count in column 3: ", df[3].value_counts())

unique values in column 2:  <bound method Series.unique of 0        0
1        0
2        0
3        0
4        0
        ..
31906    0
31907    0
31908    0
31909    0
31910    0
Name: 2, Length: 31911, dtype: int64>
value count in column 3:  2
0    31911
Name: count, dtype: int64
unique values in column 3:  <bound method Series.unique of 0         True
1         True
2         True
3         True
4         True
         ...  
31906    False
31907    False
31908    False
31909    False
31910    False
Name: 3, Length: 31911, dtype: bool>
value count in column 3:  3
False    31793
True       118
Name: count, dtype: int64


In [6]:
# columns 2 is always 0, column three has 118 isSnapshot = True values
# column 4 is a timestamp column
# determine the total time of the replay
print("column 4 range: ", df[4].min(), ", ", df[4].max())
print((df[4].max()-df[4].min())/60)

column 4 range:  1747051334.0426989 ,  1747051560.330824
3.771468750635783


In [7]:
# investigate column 5
df.nlargest(150,5)

,0,1,2,3,4,5
118,PLC/Island 1/SortingLine/SortingLine01/measurements/input/sortingLineSensInputLightBarrier,"{""value"": true, ""timestamp"": ""2025-05-12T12:00:03.850545+00:00Z""}",0,False,1.747051e+09,14.076612
14050,PLC/Island 1/MultiProcessing/MultiProcessing01/measurements/input/multiProcessingSensTurntablePosVacuum,"{""value"": true, ""timestamp"": ""2025-05-12T12:01:55.448485+00:00Z""}",0,False,1.747051e+09,7.957680
130,PLC/Island 1/events/received/message,"{""value"": ""b'{\""topicName\"":\""VacuumGripper01\"",\""timestamp\"":1.747051357799E12,\""message\"":{\""jsonType\"":\""COMMAND\"",\""type\"":\""VACUUM\"",\""outputId\"":0,\""name\"":\""SETUP\"",\""parameters\"":[]}}'"", ""timestamp"": ""2025-05-12T12:00:13.533985+00:00Z""}",0,False,1.747051e+09,7.383011
13967,PLC/Island 1/MultiProcessing/MultiProcessing01/measurements/input/multiProcessingSensTurntablePosVacuum,"{""value"": true, ""timestamp"": ""2025-05-12T12:01:45.669349+00:00Z""}",0,False,1.747051e+09,5.994579
20183,PLC/Island 1/HighBay/HighBay01/measurements/input/highbaySensCantileverBack,"{""value"": true, ""timestamp"": ""2025-05-12T12:02:36.135442+00:00Z""}",0,False,1.747052e+09,3.612863
...,...,...,...,...,...,...
8696,PLC/Island 1/VacuumGripper/VacuumGripper02/events/emitted/command_feedback,"{""value"": ""{\""jsonType\"": \""COMMAND_FEEDBACK\"", \""commandId\"": 5, \""status\"": \""DONE\"", \""info\"": \""process_sequence_CycleStep 12/12 : up \\n VacuumGripperConfig(_counterVertical=800, _counterRot=1800, _counterArm=1850, _gripperActive=False, _endVertical=False, _endRot=False, _endArm=False)\\nLast subCycleStepResult: gotoconfig VacuumGripperConfig(_counterVertical=800, _counterRot=1800, _counterArm=1850, _gripperActive=False, _endVertical=False, _endRot=False, _endArm=False) CycleStepResultEnum.DONE\""}"", ""timestamp"": ""2025-05-12T12:00:59.895566+00:00Z""}",0,False,1.747051e+09,0.044083
4341,PLC/Island 1/SortingLine/SortingLine01/measurements/input/sortingLineSensBlueLightBarrier,"{""value"": true, ""timestamp"": ""2025-05-12T12:00:29.972188+00:00Z""}",0,False,1.747051e+09,0.044056
10909,PLC/Island 1/VacuumGripper/VacuumGripper01/events/emitted/command_feedback,"{""value"": ""{\""jsonType\"": \""COMMAND_FEEDBACK\"", \""commandId\"": 8, \""status\"": \""MUST_CONTINUE\"", \""info\"": \""process_sequence_CycleStep 5/12 : VacuumGripper01.grab \\n VacuumGripperConfig(_counterVertical=1550, _counterRot=2675, _counterArm=900, _gripperActive=True, _endVertical=False, _endRot=False, _endArm=False)\\nLast subCycleStepResult: gotoconfig VacuumGripperConfig(_counterVertical=1550, _counterRot=2675, _counterArm=900, _gripperActive=False, _endVertical=False, _endRot=False, _endArm=False) CycleStepResultEnum.DONE\""}"", ""timestamp"": ""2025-05-12T12:01:13.663608+00:00Z""}",0,False,1.747051e+09,0.043881
13821,PLC/Island 1/MultiProcessing/MultiProcessing01/events/emitted/machine_feedback,"{""value"": ""{\""jsonType\"": \""MACHINE_FEEDBACK\"", \""status\"": \""INITIALIZED_ACTIVE\"", \""info\"": \""\""}"", ""timestamp"": ""2025-05-12T12:01:32.528743+00:00Z""}",0,False,1.747051e+09,0.043777


In [8]:
# column 5: higher processing time in the initial rows => powering on stations
# column 6: epoch ts

work_material = df.rename(columns={0: "topic", 1: "payload_raw", 2: "unused_flag", 3: "is_snapshot", 4: "recv_ts", 5: "proc_latency_s",})
work_material.head()

,topic,payload_raw,unused_flag,is_snapshot,recv_ts,proc_latency_s
0,PLC/Island 1/VacuumGripper/VacuumGripper01/measurements/input/vacuumSensVerticalEncoderCounter,"{""value"": 1770, ""timestamp"": ""2025-05-12T11:59:13.992112+00:00Z""}",0,True,1.747051e+09,0.040951
1,PLC/Island 1/VacuumGripper/VacuumGripper01/measurements/input/vacuumSensRotEncoderCounter,"{""value"": 2690, ""timestamp"": ""2025-05-12T11:59:13.992289+00:00Z""}",0,True,1.747051e+09,0.000252
2,PLC/Island 1/VacuumGripper/VacuumGripper01/measurements/input/vacuumSensArmEncoderCounter,"{""value"": 909, ""timestamp"": ""2025-05-12T11:59:13.992372+00:00Z""}",0,True,1.747051e+09,0.000188
3,PLC/Island 1/VacuumGripper/VacuumGripper01/measurements/input/vacuumSensVerticalEndUp,"{""value"": false, ""timestamp"": ""2025-05-12T11:59:13.992448+00:00Z""}",0,True,1.747051e+09,0.000135
4,PLC/Island 1/VacuumGripper/VacuumGripper01/measurements/input/vacuumSensRotEnd,"{""value"": false, ""timestamp"": ""2025-05-12T11:59:13.992523+00:00Z""}",0,True,1.747051e+09,0.000130


In [9]:
#define segments

segement_num = work_material["topic"].str.split("/").apply(len).value_counts()
print("number of segments in a message: ")
print(segement_num)


number of segments in a message: 
topic
7    31894
5       17
Name: count, dtype: int64


In [10]:
work_material[work_material["topic"].str.split("/").str.len() == 5].iloc[0]

topic                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                    PLC/Island 1/events/received/message
payload_raw       {"value": "b'{\"topicName\":\"VacuumGripper01\",\"timestamp\":1.74705128605E12,\"message\":{\"jsonType\":\"COMMAND\",\"type\":\"VACUUM\",\"outputId\":8,\"name\":\"MOVE\",\"parameters\":[{\"passableType\":\"POSITIONPARAMETERTHREED\",\"passable\":{\"vertical\":1300,\"horizontal\":900,\"rot\":2675,\"meaning\":\"START\"}},{\"passableType\":\"POSITIONPARAMETERTHREED\",\"passable\":{\"vertical\":950,\"horizontal\":1850,\"rot\":2010,\"meaning\":\"END\"}}]}}'"

In [11]:
# unwrap the topic into seperate columns for further inspection
PREFIX = "PLC/Island 1/"
def parse_topic(topic: str):
    rest = topic[len(PREFIX):]
    parts = rest.split("/")
    if len(parts) < 4:
        return pd.Series([None, None, parts[0] if parts else None, parts[1] if len(parts)>1 else None, None])
    station_type = parts[0] if len(parts) > 0 else None
    station_name = parts[1] if len(parts) > 1 else None
    category = parts[2] if len(parts) > 2 else None
    direction = parts[3] if len(parts) > 3 else None
    signal_name = "/".join(parts[4:]) if len(parts) > 4 else None
    return pd.Series([station_type, station_name, category, direction, signal_name])

work_material[["station_type", "station_name", "category", "direction", "signal_name"]] = (work_material["topic"].apply(parse_topic))
work_material

,topic,payload_raw,unused_flag,is_snapshot,recv_ts,proc_latency_s,station_type,station_name,category,direction,signal_name
0,PLC/Island 1/VacuumGripper/VacuumGripper01/measurements/input/vacuumSensVerticalEncoderCounter,"{""value"": 1770, ""timestamp"": ""2025-05-12T11:59:13.992112+00:00Z""}",0,True,1.747051e+09,0.040951,VacuumGripper,VacuumGripper01,measurements,input,vacuumSensVerticalEncoderCounter
1,PLC/Island 1/VacuumGripper/VacuumGripper01/measurements/input/vacuumSensRotEncoderCounter,"{""value"": 2690, ""timestamp"": ""2025-05-12T11:59:13.992289+00:00Z""}",0,True,1.747051e+09,0.000252,VacuumGripper,VacuumGripper01,measurements,input,vacuumSensRotEncoderCounter
2,PLC/Island 1/VacuumGripper/VacuumGripper01/measurements/input/vacuumSensArmEncoderCounter,"{""value"": 909, ""timestamp"": ""2025-05-12T11:59:13.992372+00:00Z""}",0,True,1.747051e+09,0.000188,VacuumGripper,VacuumGripper01,measurements,input,vacuumSensArmEncoderCounter
3,PLC/Island 1/VacuumGripper/VacuumGripper01/measurements/input/vacuumSensVerticalEndUp,"{""value"": false, ""timestamp"": ""2025-05-12T11:59:13.992448+00:00Z""}",0,True,1.747051e+09,0.000135,VacuumGripper,VacuumGripper01,measurements,input,vacuumSensVerticalEndUp
4,PLC/Island 1/VacuumGripper/VacuumGripper01/measurements/input/vacuumSensRotEnd,"{""value"": false, ""timestamp"": ""2025-05-12T11:59:13.992523+00:00Z""}",0,True,1.747051e+09,0.000130,VacuumGripper,VacuumGripper01,measurements,input,vacuumSensRotEnd
...,...,...,...,...,...,...,...,...,...,...,...
31906,PLC/Island 1/HighBay/HighBay01/measurements/input/highbaySensHorizontalEncoderCounter,"{""value"": 0, ""timestamp"": ""2025-05-12T12:03:36.039204+00:00Z""}",0,False,1.747052e+09,0.000190,HighBay,HighBay01,measurements,input,highbaySensHorizontalEncoderCounter
31907,PLC/Island 1/HighBay/HighBay01/measurements/input/highbaySensInside,"{""value"": true, ""timestamp"": ""2025-05-12T12:03:36.039347+00:00Z""}",0,False,1.747052e+09,0.000227,HighBay,HighBay01,measurements,input,highbaySensInside
31908,PLC/Island 1/HighBay/HighBay01/measurements/input/highbaySensOutside,"{""value"": true, ""timestamp"": ""2025-05-12T12:03:36.039475+00:00Z""}",0,False,1.747052e+09,0.000222,HighBay,HighBay01,measurements,input,highbaySensOutside
31909,PLC/Island 1/HighBay/HighBay01/measurements/input/highbaySensVertical,"{""value"": true, ""timestamp"": ""2025-05-12T12:03:36.039602+00:00Z""}",0,False,1.747052e+09,0.000163,HighBay,HighBay01,measurements,input,highbaySensVertical


In [12]:
#check different categories:

work_material["category"].value_counts()

category
measurements    31736
events            175
Name: count, dtype: int64

In [13]:
# find different directions per category
work_material.groupby(["category", "direction"]).size()


category      direction
events        emitted        136
              received        39
measurements  input        30451
              internal        35
              output        1250
dtype: int64

In [14]:
def parse_payload(payload):
    try:
        obj = json.loads(payload)
    except (json.JSONDecodeError, TypeError):
        return pd.Series
    return pd.Series([obj.get("value"), obj.get("timestamp")])

work_material[["value", "payload_ts"]] = work_material["payload_raw"].apply(parse_payload)
work_material[["topic", "value", "payload_ts"]].head()

#parse_failures = work_material["value"].isna().sum() - (work_material["payload_raw"] == "null").sum()
#print("rows where payload JSON failed to parse or had no 'value':", work_material["value"].isna().sum())
work_material["station_name"].isnull().any()

True

In [ ]:
# turn the value string back to bytes object, parse the json inside 
def unwrap_messagea(val):
    if not isinstance(val, str): return None
    try:
        raw_bytes = ast.literal_eval(val)
        return json.loads(raw_bytes.decode("utf-8"))
    except: return None
message_mask = work_material["topic"] == "PLC/Island 1/events/received/message"
work_material.loc[message_mask, "command_message"] = (work_material.loc[message_mask, "value"].apply(unwrap_messagea))

In [16]:
work_material[work_material["topic"].str.split("/").str.len() == 5].iloc[0]

topic                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                     PLC/Island 1/events/received/message
payload_raw        {"value": "b'{\"topicName\":\"VacuumGripper01\",\"timestamp\":1.74705128605E12,\"message\":{\"jsonType\":\"COMMAND\",\"type\":\"VACUUM\",\"outputId\":8,\"name\":\"MOVE\",\"parameters\":[{\"passableType\":\"POSITIONPARAMETERTHREED\",\"passable\":{\"vertical\":1300,\"horizontal\":900,\"rot\":2675,\"meaning\":\"START\"}},{\"passableType\":\"POSITIONPARAMETERTHREED\",\"passable\":{\"vertical\":950,\"horizontal\":1850,\"rot\":2010,\"meaning\":\"END\"}}]}}

In [17]:
# count the data types of values

valueTypes = work_material["value"].apply(lambda v: type(v).__name__) 
work_material["value_types"] = valueTypes
work_material.head()

,topic,payload_raw,unused_flag,is_snapshot,recv_ts,proc_latency_s,station_type,station_name,category,direction,signal_name,value,payload_ts,command_message,value_types
0,PLC/Island 1/VacuumGripper/VacuumGripper01/measurements/input/vacuumSensVerticalEncoderCounter,"{""value"": 1770, ""timestamp"": ""2025-05-12T11:59:13.992112+00:00Z""}",0,True,1.747051e+09,0.040951,VacuumGripper,VacuumGripper01,measurements,input,vacuumSensVerticalEncoderCounter,1770,2025-05-12T11:59:13.992112+00:00Z,NaN,int
1,PLC/Island 1/VacuumGripper/VacuumGripper01/measurements/input/vacuumSensRotEncoderCounter,"{""value"": 2690, ""timestamp"": ""2025-05-12T11:59:13.992289+00:00Z""}",0,True,1.747051e+09,0.000252,VacuumGripper,VacuumGripper01,measurements,input,vacuumSensRotEncoderCounter,2690,2025-05-12T11:59:13.992289+00:00Z,NaN,int
2,PLC/Island 1/VacuumGripper/VacuumGripper01/measurements/input/vacuumSensArmEncoderCounter,"{""value"": 909, ""timestamp"": ""2025-05-12T11:59:13.992372+00:00Z""}",0,True,1.747051e+09,0.000188,VacuumGripper,VacuumGripper01,measurements,input,vacuumSensArmEncoderCounter,909,2025-05-12T11:59:13.992372+00:00Z,NaN,int
3,PLC/Island 1/VacuumGripper/VacuumGripper01/measurements/input/vacuumSensVerticalEndUp,"{""value"": false, ""timestamp"": ""2025-05-12T11:59:13.992448+00:00Z""}",0,True,1.747051e+09,0.000135,VacuumGripper,VacuumGripper01,measurements,input,vacuumSensVerticalEndUp,False,2025-05-12T11:59:13.992448+00:00Z,NaN,bool
4,PLC/Island 1/VacuumGripper/VacuumGripper01/measurements/input/vacuumSensRotEnd,"{""value"": false, ""timestamp"": ""2025-05-12T11:59:13.992523+00:00Z""}",0,True,1.747051e+09,0.000130,VacuumGripper,VacuumGripper01,measurements,input,vacuumSensRotEnd,False,2025-05-12T11:59:13.992523+00:00Z,NaN,bool


In [18]:
# discover signal names and data types of signals

signalNames = (work_material["signal_name"]).dropna().astype(str).str.strip().drop_duplicates().sort_values().reset_index(drop=True)
signals_values = work_material[work_material["signal_name"].str.contains("command", na=False)==False][["station_type","station_name","signal_name", "value"]]
signals_values

,station_type,station_name,signal_name,value
0,VacuumGripper,VacuumGripper01,vacuumSensVerticalEncoderCounter,1770
1,VacuumGripper,VacuumGripper01,vacuumSensRotEncoderCounter,2690
2,VacuumGripper,VacuumGripper01,vacuumSensArmEncoderCounter,909
3,VacuumGripper,VacuumGripper01,vacuumSensVerticalEndUp,False
4,VacuumGripper,VacuumGripper01,vacuumSensRotEnd,False
...,...,...,...,...
31906,HighBay,HighBay01,highbaySensHorizontalEncoderCounter,0
31907,HighBay,HighBay01,highbaySensInside,True
31908,HighBay,HighBay01,highbaySensOutside,True
31909,HighBay,HighBay01,highbaySensVertical,True


In [19]:
signals_values["value_types"] = signals_values["value"].apply(lambda v: type(v).__name__) 
signals_values.groupby(["value_types", "signal_name"]).size()

value_types  signal_name                      
bool         conveyorActBackward                     3
             conveyorActForward                      3
             conveyorSensFeed                       33
             conveyorSensImpulse                    33
             conveyorSensSwap                       33
                                                  ... 
int          highbaySensVerticalEncoderCounter    1644
             vacuumSensArmEncoderCounter          2773
             vacuumSensRotEncoderCounter          2773
             vacuumSensVerticalEncoderCounter     2773
str          machine_feedback                       36
Length: 79, dtype: int64

In [20]:
int_signals = signals_values[signals_values["value_types"]=="int"]
check = (int_signals["signal_name"].apply(lambda x: x.lower().endswith("encodercounter")))
check.all()

True

In [21]:
# This means that any signal that ends with encoderCounter is an integer
# machine_feedback is the only string, anything else is a boolean


# Find min and max value for each integer value to configure initialization

numeric_mask = signals_values["value"].apply(lambda v: isinstance(v, (int,float)) and not isinstance(v, bool))
numeric_df = signals_values[numeric_mask].copy()

ranges = (numeric_df.groupby(["station_name", "signal_name"])["value"]
          .agg(min_val = "min", max_val = "max", count = "count").reset_index())
ranges["spread"] = ranges["max_val"] - ranges["min_val"]
ranges.sort_values("spread", ascending=False)


,station_name,signal_name,min_val,max_val,count,spread
3,VacuumGripper01,vacuumSensRotEncoderCounter,0,2690,1802,2690
6,VacuumGripper02,vacuumSensRotEncoderCounter,0,2284,971,2284
2,VacuumGripper01,vacuumSensArmEncoderCounter,-3,2017,1802,2020
5,VacuumGripper02,vacuumSensArmEncoderCounter,-1,1865,971,1866
4,VacuumGripper01,vacuumSensVerticalEncoderCounter,-15,1770,1802,1785
1,HighBay01,highbaySensVerticalEncoderCounter,-4,1706,1644,1710
7,VacuumGripper02,vacuumSensVerticalEncoderCounter,0,1663,971,1663
0,HighBay01,highbaySensHorizontalEncoderCounter,-4,1558,1644,1562


In [22]:
# find the initial values for the stations

snapshot = work_material[work_material["is_snapshot"] & work_material["signal_name"].notna() & (work_material["category"] == "measurements")]

for station, group in snapshot.groupby("station_name"):
    print("station: ", station)
    for _, r in group.sort_values("signal_name").iterrows():
        print(r["direction"], "     ", r["signal_name"], "     ", r["value"])
    print()
    print()

station:  ConveyorBelt01
output       conveyorActBackward       False
output       conveyorActForward       False
input       conveyorSensFeed       True
input       conveyorSensImpulse       True
input       conveyorSensSwap       True
internal       isExecuting       False


station:  HighBay01
output       highbayActCantileverBackward       False
output       highbayActCantileverForward       False
output       highbayActConveyorBackward       False
output       highbayActConveyorForward       False
output       highbayActDown       False
output       highbayActHorizontalToConveyor       False
output       highbayActHorizontalToRack       False
output       highbayActUp       False
input       highbaySensCantileverBack       True
input       highbaySensCantileverFront       False
input       highbaySensHorizontal       True
input       highbaySensHorizontalEncoderCounter       0
input       highbaySensInside       True
input       highbaySensOutside       True
input       highbaySen

In [23]:
# inpsect events

a = work_material[work_material["topic"].str.contains("events", na=False)]
a

,topic,payload_raw,unused_flag,is_snapshot,recv_ts,proc_latency_s,station_type,station_name,category,direction,signal_name,value,payload_ts,command_message,value_types
15,PLC/Island 1/VacuumGripper/VacuumGripper01/events/emitted/machine_feedback,"{""value"": ""{\""jsonType\"": \""MACHINE_FEEDBACK\"", \""status\"": \""UNINITIALIZED_IDLE\"", \""info\"": \""\""}"", ""timestamp"": ""2025-05-12T11:59:13.972460+00:00Z""}",0,True,1.747051e+09,0.000240,VacuumGripper,VacuumGripper01,events,emitted,machine_feedback,"{""jsonType"": ""MACHINE_FEEDBACK"", ""status"": ""UNINITIALIZED_IDLE"", ""info"": """"}",2025-05-12T11:59:13.972460+00:00Z,NaN,str
16,PLC/Island 1/VacuumGripper/VacuumGripper01/events/emitted/command_feedback,"{""value"": ""{\""jsonType\"": \""COMMAND_FEEDBACK\"", \""commandId\"": 8, \""status\"": \""MUST_CONTINUE\"", \""info\"": \""process_sequence_CycleStep 4/12 : VacuumGripper01.from above \\n VacuumGripperConfig(_counterVertical=1550, _counterRot=2675, _counterArm=900, _gripperActive=False, _endVertical=False, _endRot=False, _endArm=False)\\nLast subCycleStepResult: gotoconfig VacuumGripperConfig(_counterVertical=1550, _counterRot=2675, _counterArm=900, _gripperActive=False, _endVertical=False, _endRot=False, _endArm=False) CycleStepResultEnum.MUST_CONTINUE\""}"", ""timestamp"": ""2025-05-12T11:59:06.693509+00:00Z""}",0,True,1.747051e+09,0.000231,VacuumGripper,VacuumGripper01,events,emitted,command_feedback,"{""jsonType"": ""COMMAND_FEEDBACK"", ""commandId"": 8, ""status"": ""MUST_CONTINUE"", ""info"": ""process_sequence_CycleStep 4/12 : VacuumGripper01.from above \n VacuumGripperConfig(_counterVertical=1550, _counterRot=2675, _counterArm=900, _gripperActive=False, _endVertical=False, _endRot=False, _endArm=False)\nLast subCycleStepResult: gotoconfig VacuumGripperConfig(_counterVertical=1550, _counterRot=2675, _counterArm=900, _gripperActive=False, _endVertical=False, _endRot=False, _endArm=False) CycleStepResultEnum.MUST_CONTINUE""}",2025-05-12T11:59:06.693509+00:00Z,NaN,str
17,PLC/Island 1/VacuumGripper/VacuumGripper01/events/received/command,"{""value"": ""\""COMMAND VACUUM 8 MOVE [meaning START vertical 1300 rot 2675 horizontal 900, meaning END vertical 950 rot 2010 horizontal 1850]\"""", ""timestamp"": ""2025-05-12T11:59:01.805394+00:00Z""}",0,True,1.747051e+09,0.000241,VacuumGripper,VacuumGripper01,events,received,command,"""COMMAND VACUUM 8 MOVE [meaning START vertical 1300 rot 2675 horizontal 900, meaning END vertical 950 rot 2010 horizontal 1850]""",2025-05-12T11:59:01.805394+00:00Z,NaN,str
33,PLC/Island 1/VacuumGripper/VacuumGripper02/events/emitted/machine_feedback,"{""value"": ""{\""jsonType\"": \""MACHINE_FEEDBACK\"", \""status\"": \""UNINITIALIZED_IDLE\"", \""info\"": \""\""}"", ""timestamp"": ""2025-05-12T11:59:13.973977+00:00Z""}",0,True,1.747051e+09,0.000183,VacuumGripper,VacuumGripper02,events,emitted,machine_feedback,"{""jsonType"": ""MACHINE_FEEDBACK"", ""status"": ""UNINITIALIZED_IDLE"", ""info"": """"}",2025-05-12T11:59:13.973977+00:00Z,NaN,str
34,PLC/Island 1/VacuumGripper/VacuumGripper02/events/emitted/command_feedback,"{""value"": ""{\""jsonType\"": \""COMMAND_FEEDBACK\"", \""commandId\"": 5, \""status\"": \""DONE\"", \""info\"": \""process_sequence_CycleStep 12/12 : up \\n VacuumGripperConfig(_counterVertical=800, _counterRot=1800, _counterArm=1850, _gripperActive=False, _endVertical=False, _endRot=False, _endArm=False)\\nLast subCycleStepResult: gotoconfig VacuumGripperConfig(_counterVertical=800, _counterRot=1800, _counterArm=1850, _gripperActive=False, _endVertical=False, _endRot=False, _endArm=False) CycleStepResultEnum.DONE\""}"", ""timestamp"": ""2025-05-12T11:58:54.077737+00:00Z""}",0,True,1.747051e+09,0.000211,VacuumGripper,VacuumGripper02,events,emitted,command_feedback,"{""jsonType"": ""COMMAND_FEEDBACK"", ""commandId"": 5, ""status"": ""DONE"", ""info"": ""process_sequence_CycleStep 12/12 : up \n VacuumGripperConfig(_counterVertical=800, _counterRot=1800, _counterArm=1850, _gripperAc

In [86]:
a[~a["payload_raw"].str.contains("MACHINE_FEEDBACK|COMMAND_FEEDBACK", na=False)]

,topic,payload_raw,unused_flag,is_snapshot,recv_ts,proc_latency_s,station_type,station_name,category,direction,signal_name,value,payload_ts,command_message,value_types


In [87]:
a = work_material[work_material["payload_raw"].str.contains("MACHINE_FEEDBACK|COMMAND_FEEDBACK", na=False)]

records = []
for idx, row in a.iterrows():
    try:
        outer = json.loads(row["payload_raw"])
        inner = json.loads(outer["value"])
        records.append({
            "line_number": idx,
            "topic": row["topic"],
            "category": inner.get("jsonType"),
            "status": inner.get("status"),
            "info": inner.get("info", ""), 
        })
    except Exception as e:
        print("not parsed")
b = pd.DataFrame(records)
b.groupby(["category", "status"]).size()

category          status              
COMMAND_FEEDBACK  DONE                    21
                  MUST_CONTINUE           79
MACHINE_FEEDBACK  INITIALIZED_ACTIVE      10
                  INITIALIZED_IDLE        19
                  UNINITIALIZED_ACTIVE     4
                  UNINITIALIZED_IDLE       3
dtype: int64

In [88]:
# confirm similarity betewen message command events and commands

maxDifferenceInSec = 1.0
df_commands = pd.read_csv(CSV_PATH, header=None, names=["topic", "payload", "column3", "column4", "column5", "column6"])

df_commands

,topic,payload,column3,column4,column5,column6
0,PLC/Island 1/VacuumGripper/VacuumGripper01/measurements/input/vacuumSensVerticalEncoderCounter,"{""value"": 1770, ""timestamp"": ""2025-05-12T11:59:13.992112+00:00Z""}",0,True,1.747051e+09,0.040951
1,PLC/Island 1/VacuumGripper/VacuumGripper01/measurements/input/vacuumSensRotEncoderCounter,"{""value"": 2690, ""timestamp"": ""2025-05-12T11:59:13.992289+00:00Z""}",0,True,1.747051e+09,0.000252
2,PLC/Island 1/VacuumGripper/VacuumGripper01/measurements/input/vacuumSensArmEncoderCounter,"{""value"": 909, ""timestamp"": ""2025-05-12T11:59:13.992372+00:00Z""}",0,True,1.747051e+09,0.000188
3,PLC/Island 1/VacuumGripper/VacuumGripper01/measurements/input/vacuumSensVerticalEndUp,"{""value"": false, ""timestamp"": ""2025-05-12T11:59:13.992448+00:00Z""}",0,True,1.747051e+09,0.000135
4,PLC/Island 1/VacuumGripper/VacuumGripper01/measurements/input/vacuumSensRotEnd,"{""value"": false, ""timestamp"": ""2025-05-12T11:59:13.992523+00:00Z""}",0,True,1.747051e+09,0.000130
...,...,...,...,...,...,...
31906,PLC/Island 1/HighBay/HighBay01/measurements/input/highbaySensHorizontalEncoderCounter,"{""value"": 0, ""timestamp"": ""2025-05-12T12:03:36.039204+00:00Z""}",0,False,1.747052e+09,0.000190
31907,PLC/Island 1/HighBay/HighBay01/measurements/input/highbaySensInside,"{""value"": true, ""timestamp"": ""2025-05-12T12:03:36.039347+00:00Z""}",0,False,1.747052e+09,0.000227
31908,PLC/Island 1/HighBay/HighBay01/measurements/input/highbaySensOutside,"{""value"": true, ""timestamp"": ""2025-05-12T12:03:36.039475+00:00Z""}",0,False,1.747052e+09,0.000222
31909,PLC/Island 1/HighBay/HighBay01/measurements/input/highbaySensVertical,"{""value"": true, ""timestamp"": ""2025-05-12T12:03:36.039602+00:00Z""}",0,False,1.747052e+09,0.000163


In [99]:
# implement parsers

def parse_timestamp(timestamp):
    if timestamp is None:
        return pd.NaT
    timestamp = str(timestamp).replace("+00:00Z", "+00:00")
    return pd.to_datetime(timestamp, utc=True, errors="coerce")

def parse_outer_payload(payload):
    try:
        return json.loads(payload)
    except Exception:
        return None

def decode_message_value(value):
    if not isinstance(value, str):
        return None
    value = value.strip()
    if value.startswith("b'") or value.startswith('b"'):
        try:
            value = ast.literal_eval(value).decode("utf-8")
        except Exception:
            pass

    try:
        return json.loads(value)
    except Exception:
        return None

def decode_command_value(value):
    if not isinstance(value, str):
        return None
    try:
        decoded = json.loads(value)
        if isinstance(decoded, str):
            return decoded
        return str(decoded)
    except Exception:
        return value

In [100]:
message_rows = df_commands[df_commands["topic"].astype(str).str.contains("/events/received/message", na= False)].copy()
message_rows

,topic,payload,column3,column4,column5,column6
117,PLC/Island 1/events/received/message,"{""value"": ""b'{\""topicName\"":\""VacuumGripper01\"",\""timestamp\"":1.74705128605E12,\""message\"":{\""jsonType\"":\""COMMAND\"",\""type\"":\""VACUUM\"",\""outputId\"":8,\""name\"":\""MOVE\"",\""parameters\"":[{\""passableType\"":\""POSITIONPARAMETERTHREED\"",\""passable\"":{\""vertical\"":1300,\""horizontal\"":900,\""rot\"":2675,\""meaning\"":\""START\""}},{\""passableType\"":\""POSITIONPARAMETERTHREED\"",\""passable\"":{\""vertical\"":950,\""horizontal\"":1850,\""rot\"":2010,\""meaning\"":\""END\""}}]}}'"", ""timestamp"": ""2025-05-12T11:59:01.766666+00:00Z""}",0,True,1.747051e+09,0.000103
130,PLC/Island 1/events/received/message,"{""value"": ""b'{\""topicName\"":\""VacuumGripper01\"",\""timestamp\"":1.747051357799E12,\""message\"":{\""jsonType\"":\""COMMAND\"",\""type\"":\""VACUUM\"",\""outputId\"":0,\""name\"":\""SETUP\"",\""parameters\"":[]}}'"", ""timestamp"": ""2025-05-12T12:00:13.533985+00:00Z""}",0,False,1.747051e+09,7.383011
2294,PLC/Island 1/events/received/message,"{""value"": ""b'{\""topicName\"":\""VacuumGripper02\"",\""timestamp\"":1.747051365038E12,\""message\"":{\""jsonType\"":\""COMMAND\"",\""type\"":\""VACUUM\"",\""outputId\"":1,\""name\"":\""SETUP\"",\""parameters\"":[]}}'"", ""timestamp"": ""2025-05-12T12:00:20.755295+00:00Z""}",0,False,1.747051e+09,0.004904
4118,PLC/Island 1/events/received/message,"{""value"": ""b'{\""topicName\"":\""MultiProcessing01\"",\""timestamp\"":1.747051371039E12,\""message\"":{\""jsonType\"":\""COMMAND\"",\""type\"":\""MULTIPROCESSING\"",\""outputId\"":2,\""name\"":\""SETUP\"",\""parameters\"":[]}}'"", ""timestamp"": ""2025-05-12T12:00:26.756034+00:00Z""}",0,False,1.747051e+09,0.000233
4120,PLC/Island 1/events/received/message,"{""value"": ""b'{\""topicName\"":\""HighBay01\"",\""timestamp\"":1.747051371041E12,\""message\"":{\""jsonType\"":\""COMMAND\"",\""type\"":\""WAREHOUSE\"",\""outputId\"":3,\""name\"":\""SETUP\"",\""parameters\"":[]}}'"", ""timestamp"": ""2025-05-12T12:00:26.758890+00:00Z""}",0,False,1.747051e+09,0.000212
4124,PLC/Island 1/events/received/message,"{""value"": ""b'{\""topicName\"":\""SortingLine01\"",\""timestamp\"":1.747051371042E12,\""message\"":{\""jsonType\"":\""COMMAND\"",\""type\"":\""SORTING\"",\""outputId\"":4,\""name\"":\""EJECT\"",\""parameters\"":[{\""passableType\"":\""COLOR\"",\""passable\"":{\""color\"":\""RED\""}}]}}'"", ""timestamp"": ""2025-05-12T12:00:26.761788+00:00Z""}",0,False,1.747051e+09,0.000351
4476,PLC/Island 1/events/received/message,"{""value"": ""b'{\""topicName\"":\""VacuumGripper02\"",\""timestamp\"":1.747051376513E12,\""message\"":{\""jsonType\"":\""COMMAND\"",\""type\"":\""VACUUM\"",\""outputId\"":5,\""name\"":\""MOVE\"",\""parameters\"":[{\""passableType\"":\""POSITIONPARAMETERTHREED\"",\""passable\"":{\""vertical\"":1400,\""horizontal\"":1225,\""rot\"":2275,\""meaning\"":\""START\""}},{\""passableType\"":\""POSITIONPARAMETERTHREED\"",\""passable\"":{\""vertical\"":1050,\""horizontal\"":1850,\""rot\"":1800,\""meaning\"":\""END\""}}]}}'"", ""timestamp"": ""2025-05-12T12:00:32.232105+00:00Z""}",0,False,1.747051e+09,0.034082
8712,PLC/Island 1/events/received/message,"{""value"": ""b'{\""topicName\"":\""ConveyorBelt01\"",\""timestamp\"":1.747051404222E12,\""message\"":{\""jsonType\"":\""COMMAND\"",\""type\"":\""CONVEYOR\"",\""outputId\"":6,\""name\"":\""MOVE_TO_SENSOR\"",\""parameters\"":[{\""passableType\"":\""DIRECTION\"",\""passable\"":{\""direction\"":\""FORWARD\""}}]}}'"", ""timestamp"": ""2025-05-12T12:00:59.938353+00:00Z""}",0,False,1.747051e+09,0.032600
8713,PLC/Island 1/events/received/message,"{""value"": ""b'{\""topicName\"":\""VacuumGripper01\"",\""timestamp\"":1.747051404223E12,\""message\"":{\""jsonType\"":\""COMMAND\"",\""type\"":\""VACUUM\"",\""outputId\"":7,\""name\"":\""GO_TO_POSITION\"",\""parameters\"":[{\""passableType\"":\""POSITIONPARAMETERTHREED\"",\""passable\"":{\""vertical\"":0,\""horizontal\"":0,\""rot\"":2675,

In [101]:
message_commands = []

for ind, row in message_rows.iterrows():
    outer  = parse_outer_payload(row["payload"])
    if(outer is None):
        continue
    inner = decode_message_value(outer.get("value"))
    if (inner is None):
        continue
    message = inner.get("message")
    if not isinstance(message, dict):
        continue
    if(str(message.get("jsonType", "")).upper()!="COMMAND"):
        continue
    message_commands.append({"row":ind, "station":inner.get("topicName"), "command_id": str(message.get("outputId")), "command_type": str(message.get("type", "")), "command_name": str(message.get("name", "")), "parameters": message.get("parameters"), "timestamp": parse_timestamp(outer.get("timestamp")), "topic": row["topic"]})

for i in message_commands:
    print("msg:")
    print(i)
    print()

msg:
{'row': 117, 'station': 'VacuumGripper01', 'command_id': '8', 'command_type': 'VACUUM', 'command_name': 'MOVE', 'parameters': [{'passableType': 'POSITIONPARAMETERTHREED', 'passable': {'vertical': 1300, 'horizontal': 900, 'rot': 2675, 'meaning': 'START'}}, {'passableType': 'POSITIONPARAMETERTHREED', 'passable': {'vertical': 950, 'horizontal': 1850, 'rot': 2010, 'meaning': 'END'}}], 'timestamp': Timestamp('2025-05-12 11:59:01.766666+0000', tz='UTC'), 'topic': 'PLC/Island 1/events/received/message'}

msg:
{'row': 130, 'station': 'VacuumGripper01', 'command_id': '0', 'command_type': 'VACUUM', 'command_name': 'SETUP', 'parameters': [], 'timestamp': Timestamp('2025-05-12 12:00:13.533985+0000', tz='UTC'), 'topic': 'PLC/Island 1/events/received/message'}

msg:
{'row': 2294, 'station': 'VacuumGripper02', 'command_id': '1', 'command_type': 'VACUUM', 'command_name': 'SETUP', 'parameters': [], 'timestamp': Timestamp('2025-05-12 12:00:20.755295+0000', tz='UTC'), 'topic': 'PLC/Island 1/events/r

In [107]:
# extract commands
command_rows = df_commands[df_commands["topic"].astype(str).str.contains("/events/received/command", na = False)].copy()
station_commands = []

for ind, row in command_rows.iterrows():
    outer2  = parse_outer_payload(row["payload"])
    if(outer2 is None):
        continue
    command_text = decode_command_value(outer2.get("value"))
    if (command_text is None):
        continue
    command_text = command_text.strip()
    parts = command_text.split(None, 4)
    if(len(parts)<4): continue
    if parts[0].upper() != "COMMAND": continue
    topic_parts = str(row["topic"]).split("/")
    station = (topic_parts[3] if len(topic_parts)>3 else None)

    station_commands.append({"row":ind, "station":station, "command_type":parts[1], "command_id": parts[2], "command_name": parts[3], "parameters": parts[4] if len(parts)>4 else "", "timestamp": parse_timestamp(outer2.get("timestamp")), "topic": row["topic"], "full_command":command_text})

for i in station_commands:
    print("msg:")
    print(i)
    print()


msg:
{'row': 17, 'station': 'VacuumGripper01', 'command_type': 'VACUUM', 'command_id': '8', 'command_name': 'MOVE', 'parameters': '[meaning START vertical 1300 rot 2675 horizontal 900, meaning END vertical 950 rot 2010 horizontal 1850]', 'timestamp': Timestamp('2025-05-12 11:59:01.805394+0000', tz='UTC'), 'topic': 'PLC/Island 1/VacuumGripper/VacuumGripper01/events/received/command', 'full_command': 'COMMAND VACUUM 8 MOVE [meaning START vertical 1300 rot 2675 horizontal 900, meaning END vertical 950 rot 2010 horizontal 1850]'}

msg:
{'row': 35, 'station': 'VacuumGripper02', 'command_type': 'VACUUM', 'command_id': '5', 'command_name': 'MOVE', 'parameters': '[meaning START vertical 1400 rot 2275 horizontal 1150, meaning END vertical 1050 rot 1800 horizontal 1850]', 'timestamp': Timestamp('2025-05-12 11:58:26.703487+0000', tz='UTC'), 'topic': 'PLC/Island 1/VacuumGripper/VacuumGripper02/events/received/command', 'full_command': 'COMMAND VACUUM 5 MOVE [meaning START vertical 1400 rot 2275 ho

In [ ]:
# search for matches
results = []

used_command_rows = []

for message in message_commands:
    matchings = []
    for command in station_commands:
        if(command["row"] in used_command_rows): continue
        if(message["command_id"]!=command["command_id"]): continue
        if(message["command_type"].upper()!=command["command_type"].upper()): continue
        if(message["command_name"].upper()!=command["command_name"].upper()): continue
        if(pd.isna(message["timestamp"]) or pd.isna(command["timestamp"])): continue
        difference = abs((message["timestamp"]-command["timestamp"]).total_seconds())
        if (difference<=maxDifferenceInSec): matchings.append((difference, command))

    if(matchings):
        matchings.sort(key=lambda x: x[0])
        difference, command = matchings[0]
        used_command_rows.append(command["row"])
        message_parameters = json.dumps(message["parameters"], ensure_ascii=False)
        command_parameters = command["parameters"]
        parameters_identical = (message_parameters == command_parameters)
        results.append({"status": "MATCH", "message_row":message["row"], "command_row": command["row"], "station": message["station"], "command_id": message["command_id"], "command_type": message["command_type"], "command_name": message["command_name"], "message_timestamp": message["timestamp"], "command_timestamp": command["timestamp"], "time_dif": round(difference*1000, 3), "parameters_identical": parameters_identical, "message_parameters": message_parameters, "command_parameters": command_parameters, "message_topic": message["topic"], "command_topic": command["topic"], "full_command": command["full_command"]})
    else:
        results.append({"status": "NO_MATCH", "message_row":message["row"], "command_row": None, "station": message["station"], "command_id": message["command_id"], "command_type": message["command_type"], "command_name": message["command_name"], "message_timestamp": message["timestamp"], "command_timestamp": None, "time_dif": None, "parameters_identical": False, "message_parameters": json.dumps(message["parameters"], ensure_ascii=False), "command_parameters": None, "message_topic": message["topic"],  "command_topic": None, "full_command": None})

df_results = pd.DataFrame(results)
df_results


,status,message_row,command_row,station,command_id,command_type,command_name,message_timestamp,command_timestamp,time_dif,parameters_identical,message_parameters,command_parameters,message_topic,command_topic,full_command
0,MATCH,117,17,VacuumGripper01,8,VACUUM,MOVE,2025-05-12 11:59:01.766666+00:00,2025-05-12 11:59:01.805394+00:00,38.728,False,"[{""passableType"": ""POSITIONPARAMETERTHREED"", ""passable"": {""vertical"": 1300, ""horizontal"": 900, ""rot"": 2675, ""meaning"": ""START""}}, {""passableType"": ""POSITIONPARAMETERTHREED"", ""passable"": {""vertical"": 950, ""horizontal"": 1850, ""rot"": 2010, ""meaning"": ""END""}}]","[meaning START vertical 1300 rot 2675 horizontal 900, meaning END vertical 950 rot 2010 horizontal 1850]",PLC/Island 1/events/received/message,PLC/Island 1/VacuumGripper/VacuumGripper01/events/received/command,"COMMAND VACUUM 8 MOVE [meaning START vertical 1300 rot 2675 horizontal 900, meaning END vertical 950 rot 2010 horizontal 1850]"
1,MATCH,130,131,VacuumGripper01,0,VACUUM,SETUP,2025-05-12 12:00:13.533985+00:00,2025-05-12 12:00:13.540618+00:00,6.633,True,[],[],PLC/Island 1/events/received/message,PLC/Island 1/VacuumGripper/VacuumGripper01/events/received/command,COMMAND VACUUM 0 SETUP []
2,MATCH,2294,2301,VacuumGripper02,1,VACUUM,SETUP,2025-05-12 12:00:20.755295+00:00,2025-05-12 12:00:20.763236+00:00,7.941,True,[],[],PLC/Island 1/events/received/message,PLC/Island 1/VacuumGripper/VacuumGripper02/events/received/command,COMMAND VACUUM 1 SETUP []
3,MATCH,4118,4125,MultiProcessing01,2,MULTIPROCESSING,SETUP,2025-05-12 12:00:26.756034+00:00,2025-05-12 12:00:26.762864+00:00,6.830,True,[],[],PLC/Island 1/events/received/message,PLC/Island 1/MultiProcessing/MultiProcessing01/events/received/command,COMMAND MULTIPROCESSING 2 SETUP []
4,MATCH,4120,4133,HighBay01,3,WAREHOUSE,SETUP,2025-05-12 12:00:26.758890+00:00,2025-05-12 12:00:26.798599+00:00,39.709,True,[],[],PLC/Island 1/events/received/message,PLC/Island 1/HighBay/HighBay01/events/received/command,COMMAND WAREHOUSE 3 SETUP []
5,MATCH,4124,4136,SortingLine01,4,SORTING,EJECT,2025-05-12 12:00:26.761788+00:00,2025-05-12 12:00:26.846162+00:00,84.374,False,"[{""passableType"": ""COLOR"", ""passable"": {""color"": ""RED""}}]",[<Color.RED: 1>],PLC/Island 1/events/received/message,PLC/Island 1/SortingLine/SortingLine01/events/received/command,COMMAND SORTING 4 EJECT [<Color.RED: 1>]
6,MATCH,4476,4478,VacuumGripper02,5,VACUUM,MOVE,2025-05-12 12:00:32.232105+00:00,2025-05-12 12:00:32.265818+00:00,33.713,False,"[{""passableType"": ""POSITIONPARAMETERTHREED"", ""passable"": {""vertical"": 1400, ""horizontal"": 1225, ""rot"": 2275, ""meaning"": ""START""}}, {""passableType"": ""POSITIONPARAMETERTHREED"", ""passable"": {""vertical"": 1050, ""horizontal"": 1850, ""rot"": 1800, ""meaning"": ""END""}}]","[meaning START vertical 1400 rot 2275 horizontal 1225, meaning END vertical 1050 rot 1800 horizontal 1850]",PLC/Island 1/events/received/message,PLC/Island 1/VacuumGripper/VacuumGripper02/events/received/command,"COMMAND VACUUM 5 MOVE [meaning START vertical 1400 rot 2275 horizontal 1225, meaning END vertical 1050 rot 1800 horizontal 1850]"
7,MATCH,8712,8721,ConveyorBelt01,6,CONVEYOR,MOVE_TO_SENSOR,2025-05-12 12:00:59.938353+00:00,2025-05-12 12:00:59.975931+00:00,37.578,False,"[{""passableType"": ""DIRECTION"", ""passable"": {""direction"": ""FORWARD""}}]",[<Direction.FORWARD: 1>],PLC/Island 1/events/received/message,PLC/Island 1/ConveyorBelt/ConveyorBelt01/events/received/command,COMMAND CONVEYOR 6 MOVE_TO_SENSOR [<Direction.FORWARD: 1>]
8,MATCH,8713,8733,VacuumGripper01,7,VACUUM,GO_TO_POSITION,2025-05-12 12:00:59.941170+00:00,2025-05-12 12:01:00.035714+00:00,94.544,False,"[{""passableType"": ""POSITIONPARAMETERTHREED"", ""passable"": {""vertical"": 0, ""horizontal"": 0, ""rot"": 2675, ""meaning"": ""START""}}]",[meaning START vertical 0 rot 2675 horizontal 0],PLC/Island 1/events/received/message,PLC/Island 1/VacuumGripper/VacuumGripper01/events/received/command,COMMAN